In [ ]:
import cv2
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from ultralytics import YOLO
from pathlib import Path


dataset_path = r'D:\700\train'
image_path = Path(dataset_path) / 'images'
label_path = Path(dataset_path) / 'labels'


yolo_model = YOLO('yolov8n.pt')




def detect_hands(image):
    results = yolo_model(image, verbose=False)  # Run the model on the image
    boxes = []
    labels = []
    for result in results:
        for box in result.boxes:  
            if box.cls == 0:  #'0' for hands
                x1, y1, x2, y2 = box.xyxy[0]  
                x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])  
                boxes.append([x1, y1, x2, y2])
                labels.append("Hand")  
    return boxes, labels

def extract_hand_features(image, boxes):
    hand_images = []
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cropped_hand = image[y1:y2, x1:x2]
        # Preprocess the cropped hand (resize and convert to grayscale or RGB)
        cropped_hand = cv2.resize(cropped_hand, (64, 64))  
        cropped_hand = cv2.cvtColor(cropped_hand, cv2.COLOR_BGR2GRAY)  
        cropped_hand = cropped_hand / 255.0  
        flattened_hand = cropped_hand.flatten()  
        hand_images.append(flattened_hand)
    return hand_images


def load_dataset():
    features = []
    labels = []
    for image_file in os.listdir(image_path):
        if image_file.endswith('.jpg') or image_file.endswith('.png'):
            # Load image and label
            image = cv2.imread(str(image_path / image_file))
            label_file = label_path / (image_file.replace('.jpg', '.txt').replace('.png', '.txt'))
            with open(label_file, 'r') as f:
                label = f.readlines()[0].strip()  

            # Detect hands and extract features
            boxes, _ = detect_hands(image)
            hand_features = extract_hand_features(image, boxes)

            if hand_features:
                features.extend(hand_features)
                labels.extend([label] * len(hand_features))  

    return np.array(features), np.array(labels)

#Train Random Forest model
features, labels = load_dataset()
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


import joblib
joblib.dump(rf_model, "rf_gesture_model.pkl")



y_pred = rf_model.predict(X_test)




def predict_gestures_in_folder(folder_path):
    folder_path = Path(folder_path)
    all_predictions = {}

    for image_file in folder_path.iterdir():
        if image_file.suffix.lower() in ['.jpg', '.png']:
            image = cv2.imread(str(image_file))
            boxes, labels = detect_hands(image)

            if not boxes:
                print(f"No hands detected in: {image_file.name}")
                continue

            hand_features = extract_hand_features(image, boxes)

            if hand_features:
                predictions = rf_model.predict(hand_features)
                all_predictions[image_file.name] = predictions

                # Draw bounding boxes and labels
                for box, label, pred in zip(boxes, labels, predictions):
                    x1, y1, x2, y2 = map(int, box)
                    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(image, f"{label}: {pred}", (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

                # Show the image
                cv2.imshow("Predicted Gestures", image)
                cv2.waitKey(10000)  
                cv2.destroyAllWindows()

    return all_predictions


val_dataset_path = r"D:\700\val\images"
predicted_gesture_results = predict_gestures_in_folder(val_dataset_path)

if predicted_gesture_results:
    print("Predictions completed for all validation images:")
    for image_name, gestures in predicted_gesture_results.items():
        print(f"{image_name}:{gestures}  ")
else:
    print("No gestures predicted in the validation dataset.")